# Student Results Report

*Combine CSV loading, column inspection, Boolean selection, and derived decisions in a saved report.*


## Before You Run

In **Add files**, select `sample_scores.csv` from this lesson's supplied assets.
Enter `assets/07_student_results` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/07_student_results/sample_scores.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/07_student_results/` in the file browser.


## Student Records

A student passes when the score is at least 85 and attendance is at
least 90. Both thresholds include equality. The report records a decision
for every student and saves the qualifying records to a separate CSV.

The supplied `sample_scores.csv` contains these four students:

| Name | Department | Score | Attendance |
|---|---|---:|---:|
| Minji | Data | 88 | 95 |
| Joon | AI | 76 | 82 |
| Hana | Data | 93 | 98 |
| Taeho | Design | 85 | 90 |


## Imports and File Locations

Set the input and output locations once. Every call to the workflow loads
the original CSV and replaces the same output, so repeated runs do not add rows.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "07_student_results")
output_directory = os.path.join("outputs", "07_student_results")
input_path = os.path.join(input_directory, "sample_scores.csv")
output_path = os.path.join(output_directory, "qualified_students.csv")
os.makedirs(output_directory, exist_ok=True)


## Calculate Decisions and Select Records

Keep the input table intact by building a copy. The comparison Series
share the table's index, so each decision stays with the correct student.
Return both the complete decision table and the selected report.


In [2]:
def prepare_results(scores):
    results = scores.copy()
    results["passed"] = (results["score"] >= 85) & (results["attendance"] >= 90)
    selected = results.loc[
        results["passed"],
        ["name", "department", "score", "attendance", "passed"],
    ].copy()
    return results, selected


## Save and Reopen the Selected Table

The CSV stores all report fields, including the Boolean decision.
Reopening it provides the records that a later reader would receive.


In [3]:
def save_and_reopen(selected, destination):
    selected.to_csv(destination, index=False, encoding="utf-8")
    return pd.read_csv(destination, encoding="utf-8")


## Run the Workflow

`main()` connects loading, processing, saving, and reporting. The saved
table has a new default index, so compare it with a reset-index version
of the selection. Return the result tables for further inspection.


In [4]:
def main():
    scores = pd.read_csv(input_path, encoding="utf-8")
    print("Loaded scores:")
    print(scores)
    print("Input shape:", scores.shape)

    results, selected = prepare_results(scores)
    print("Decision table:")
    print(results)
    print("Overall score mean:", scores["score"].mean())
    print("Qualifying names:", selected["name"].tolist())
    print("Qualifying count:", len(selected))

    reopened = save_and_reopen(selected, output_path)
    print("Saved file:", output_path)
    print("Reopened report:")
    print(reopened)
    print("Saved report matches:", reopened.equals(selected.reset_index(drop=True)))
    print("Input has its original columns:", "passed" not in scores.columns)
    return results, reopened


results_df, saved_report_df = main()


Loaded scores:
    name department  score  attendance
0  Minji       Data     88          95
1   Joon         AI     76          82
2   Hana       Data     93          98
3  Taeho     Design     85          90
Input shape: (4, 4)
Decision table:
    name department  score  attendance  passed
0  Minji       Data     88          95    True
1   Joon         AI     76          82   False
2   Hana       Data     93          98    True
3  Taeho     Design     85          90    True
Overall score mean: 85.5
Qualifying names: ['Minji', 'Hana', 'Taeho']
Qualifying count: 3
Saved file: outputs/07_student_results/qualified_students.csv
Reopened report:
    name department  score  attendance  passed
0  Minji       Data     88          95    True
1   Hana       Data     93          98    True
2  Taeho     Design     85          90    True
Saved report matches: True
Input has its original columns: True


## Result Analysis

The four students have an overall score mean of **85.5**. **Minji, Hana,
and Taeho** satisfy both requirements; Joon satisfies neither. Taeho's score 85 and attendance 90 meet the inclusive thresholds.

The saved report contains **3 rows and 5 columns**. `Saved report matches`
is `True`, and the input retains its original four columns. The output is
`outputs/07_student_results/qualified_students.csv`; rerunning the workflow
replaces that file with the same three records.
